In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print(np.__version__)
import torch
print(torch.__version__)
import torch.nn as nn
import sys
print(sys.path)


2.4.4


## 1) Generate One COB-ODT Dataset

In [5]:
import numpy as np

from first_experiment.odt import generate_cob_odt_data

# Paper-1-style defaults for milestone 1
x, y, tree, meta = generate_cob_odt_data(
    num_data=30_000,
    dim=100,
    depth=4,
    seed=365,
    threshold=0.0,
)

print("x shape:", x.shape)
print("y shape:", y.shape)
print("meta:", meta)

x shape: (30000, 100)
y shape: (30000,)
meta: {'num_requested': 30000, 'num_kept': 30000, 'num_pruned': 0, 'dim': 100, 'depth': 4, 'num_internal_nodes': 15, 'num_leaf_nodes': 16, 'threshold': 0.0, 'seed': 365}


## 2) Check Overall Label Balance

In [7]:
unique_labels, label_counts = np.unique(y, return_counts=True)
label_balance = dict(zip(unique_labels.tolist(), label_counts.tolist()))

print("Label counts:", label_balance)
print("Label fractions:", {k: v / len(y) for k, v in label_balance.items()})
print("Absolute imbalance:", abs(label_balance.get(-1, 0) - label_balance.get(1, 0)))

Label counts: {-1: 15110, 1: 14890}
Label fractions: {-1: 0.5036666666666667, 1: 0.49633333333333335}
Absolute imbalance: 220


## 3) Compute Samples Per Leaf

In [8]:
# Recompute leaf id for each sample using the same tree traversal logic.
# Internal nodes are indexed [0, ..., 2^depth-2]
# Leaf nodes are indexed [2^depth-1, ..., 2^(depth+1)-2]

depth = meta["depth"]
num_internal = meta["num_internal_nodes"]

margins = x @ tree.w_list.T + tree.b_list
curr_index = np.zeros(x.shape[0], dtype=np.int64)

for _ in range(depth):
    decision = margins[np.arange(x.shape[0]), curr_index]
    went_right = (decision > 0).astype(np.int64)
    curr_index = 2 * curr_index + 1 + went_right

leaf_node_ids = curr_index                  # tree-global node ids
leaf_offsets = leaf_node_ids - num_internal # 0..(2^depth-1), useful for indexing leaf arrays

leaf_ids, leaf_counts = np.unique(leaf_node_ids, return_counts=True)
print("Number of occupied leaves:", len(leaf_ids), "out of", meta["num_leaf_nodes"])
print("Per-leaf counts:")
for leaf_id, count in zip(leaf_ids, leaf_counts):
    print(f"  leaf {leaf_id}: {count}")

Number of occupied leaves: 16 out of 16
Per-leaf counts:
  leaf 15: 1872
  leaf 16: 1857
  leaf 17: 1887
  leaf 18: 1891
  leaf 19: 1879
  leaf 20: 1880
  leaf 21: 1933
  leaf 22: 1811
  leaf 23: 1800
  leaf 24: 1876
  leaf 25: 1922
  leaf 26: 1874
  leaf 27: 1944
  leaf 28: 1786
  leaf 29: 1873
  leaf 30: 1915
